In [43]:
import sys
import os
import random
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from typing import List, Dict, Tuple, Optional, Type, Union

sys.path.append(os.path.abspath("../src"))

from abstractions3d.primitives.shapes import Box3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.nodes import Rect3D, Move3D, Union3D, SymTrans3D, SymRef3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D


In [44]:
def sample_uniform(low: float, high: float) -> float:
    return random.uniform(low, high)

def generate_dataset(num_samples: int = 50) -> List[Shape3D]:
    dataset = []
    for _ in range(num_samples):
        table = Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        )
        dataset.append(table)

        chair = Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(chair)

        bench = Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(0.5, 1.5),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(bench)
    return dataset

dataset = generate_dataset()
print(f"Generated {len(dataset)} shapes")

Generated 150 shapes


In [45]:
shape = random.choice(dataset)
print("Random shape from dataset:")
visualize_boxes_3d(shape.get_box3d_list())

Random shape from dataset:


Output()

In [64]:
def add_dicts(d1: Dict, d2: Dict) -> Dict:
    for k in d2:
        d1[k].extend(d2[k])
    return d1

def get_singletons(shapes: Union[Shape3D, List[Shape3D]]) -> Dict[Tuple[str,int],List[Tuple[float,...]]]:
    singletons = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            singletons = add_dicts(singletons,get_singletons(s))
        return singletons
    t, params = shapes.param_tuple()
    floats = tuple(p for p in params if isinstance(p,(float,int)))
    if floats:
        singletons[(t.__name__, len(floats))].append(floats)
    for c in getattr(shapes,'children',[]):
        singletons = add_dicts(singletons,get_singletons(c))
    return singletons

def get_pairs(shapes: Union[Shape3D, List[Shape3D]]) -> Dict[Tuple[str,int],List[Tuple[float,...]]]:
    pairs = defaultdict(list)
    if isinstance(shapes,list):
        for s in shapes:
            pairs = add_dicts(pairs,get_pairs(s))
        return pairs
    if isinstance(shapes, Union3D):
        t, (c1, c2) = shapes.param_tuple()
        p1 = tuple(p for p in c1.param_tuple()[1] if isinstance(p,(float,int)))
        p2 = tuple(p for p in c2.param_tuple()[1] if isinstance(p,(float,int)))
        key = (f"Union3D({type(c1).__name__},{type(c2).__name__})", len(p1)+len(p2))
        pairs[key].append(p1+p2)
        for c in shapes.children:
            pairs = add_dicts(pairs,get_pairs(c))
    else:
        for c in getattr(shapes,'children',[]):
            pairs = add_dicts(pairs,get_pairs(c))
    return pairs

In [65]:
class AbstractionNode(Shape3D):
    def __init__(self, type_list: List[Type[Shape3D]], latent: torch.Tensor, model: Optional[nn.Module] = None):
        super().__init__(children=[])
        self.type_list = type_list
        self.latent = latent
        self.model = model

    def expand(self) -> Shape3D:
        safe_type_list = [Shape3D if t==AbstractionNode else t for t in self.type_list]
        if self.model is None:
            param_list = self.latent.tolist()
        else:
            self.model.eval()
            with torch.no_grad():
                latent = self.latent
                if latent.numel() != self.model.decoder[0].in_features:
                    latent = latent[:self.model.decoder[0].in_features]
                param_list = self.model.decoder(latent[None,:])[0].tolist()
        return instantiate_3d(safe_type_list, param_list)

    def get_box3d_list(self) -> List[Box3D]:
        return self.expand().get_box3d_list()

    def __str__(self):
        return f"AbstractionNode(type_list={[t.__name__ for t in self.type_list]}, latent={self.latent.tolist()})"

In [66]:
class Autoencoder(nn.Module):
    def __init__(self,input_dim:int,latent_dim:int):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim,64), nn.ReLU(), nn.Linear(64,latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim,64), nn.ReLU(), nn.Linear(64,input_dim))
    def forward(self,x:torch.Tensor):
        z = self.encoder(x)
        return z,self.decoder(z)

In [67]:
def get_direct_float_params(shape: Shape3D) -> List[float]:
    """
    Return the direct float parameters of a shape.
    Works for Shape3D and AbstractionNode.
    """
    if isinstance(shape, AbstractionNode):
        return shape.latent.tolist()
    else:
        _, params = shape.param_tuple()
        return [p for p in params if isinstance(p,(float,int))]

In [68]:
def find_abstractions(structures: Dict[Tuple[str,int],List[Tuple[float,...]]],
                      retrain_iterations: int = 10,
                      error_threshold: float = 0.05) -> Tuple[Dict, Dict]:
    models = {}
    losses = {}
    for key, params in structures.items():
        if not params: continue
        float_params = [list(p) for p in params if any(isinstance(x,(float,int)) for x in p)]
        if not float_params: continue
        num_float = len(float_params[0])
        if num_float < 2: continue

        train_tensor = torch.tensor(float_params,dtype=torch.float32)
        train_dl = DataLoader(TensorDataset(train_tensor), batch_size=32, shuffle=True)
        model = Autoencoder(input_dim=num_float, latent_dim=max(1,num_float-1))
        optimizer = AdamW(model.parameters(), lr=1e-3)
        loss_fn = lambda pred,target: torch.mean(torch.max(torch.abs(pred-target),dim=-1)[0])

        model.train()
        losses[key] = []
        for _ in range(retrain_iterations):
            batch_losses = []
            for batch in train_dl:
                x = batch[0]
                optimizer.zero_grad()
                _, x_rec = model(x)
                loss = loss_fn(x_rec,x)
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            losses[key].append(sum(batch_losses)/len(batch_losses))
        models[key] = model
        print(f"Trained {key}, final loss: {losses[key][-1]:.4f}")
    return models, losses

In [69]:
def integrate_abstractions(shape: Shape3D, models: Dict, error_threshold: float = 0.05, stats: Optional[Dict] = None) -> Shape3D:
    if stats is None: stats = {'replaced':0}

    if isinstance(shape, Union3D):
        new_c1 = integrate_abstractions(shape.children[0], models, error_threshold, stats)
        new_c2 = integrate_abstractions(shape.children[1], models, error_threshold, stats)

        float_params = get_direct_float_params(new_c1) + get_direct_float_params(new_c2)
        key = (f"Union3D({type(new_c1).__name__},{type(new_c2).__name__})", len(float_params))

        if key in models and float_params:
            model = models[key]
            input_tensor = torch.tensor(float_params)[None,:]
            with torch.no_grad():
                _, recon = model(input_tensor)
                error = torch.max(torch.abs(recon - input_tensor)).item()
            if error < error_threshold:
                stats['replaced'] += 1
                return AbstractionNode([Union3D,type(new_c1),type(new_c2)],
                                       torch.tensor(float_params,dtype=torch.float32),
                                       model)
        return Union3D(new_c1,new_c2)

    type_, params = shape.param_tuple()
    float_params = get_direct_float_params(shape)
    key = (type_.__name__, len(float_params))
    if key in models and float_params:
        model = models[key]
        input_tensor = torch.tensor(float_params)[None,:]
        with torch.no_grad():
            _, recon = model(input_tensor)
            error = torch.max(torch.abs(recon - input_tensor)).item()
        if error < error_threshold:
            stats['replaced'] += 1
            return AbstractionNode([type_], torch.tensor(float_params,dtype=torch.float32), model)

    # Return fresh instance to avoid modifying original
    new_children = [integrate_abstractions(c, models, error_threshold, stats) for c in getattr(shape,'children',[])]
    return type_(*params) if not new_children else type_(*params)

In [71]:
singletons = get_singletons(dataset)
pairs = get_pairs(dataset)
structures = add_dicts(singletons,pairs)
print("Extracted structures:", structures.keys())

models, losses = find_abstractions(structures)

abstracted_dataset = []
stats_list = []
for shape in dataset:
    stats = {'replaced':0}
    abstracted = integrate_abstractions(shape, models, error_threshold=0.05, stats=stats)
    abstracted_dataset.append(abstracted)
    stats_list.append(stats)

total_replaced = sum(s['replaced'] for s in stats_list)
print(f"Total nodes replaced: {total_replaced}, avg per shape: {total_replaced/len(dataset):.2f}")

for i,s in enumerate(stats_list):
    if s['replaced']>0:
        print(f"Shape {i} contains {s['replaced']} abstraction nodes")

Extracted structures: dict_keys([('Move3D', 3), ('Rect3D', 3), ('Union3D(Move3D,SymRef3D)', 3), ('Union3D(Move3D,Union3D)', 3), ('Union3D(SymRef3D,Move3D)', 3)])
Trained ('Move3D', 3), final loss: 0.2545
Trained ('Rect3D', 3), final loss: 0.4643
Trained ('Union3D(Move3D,SymRef3D)', 3), final loss: 1.5823
Trained ('Union3D(Move3D,Union3D)', 3), final loss: 0.0907
Trained ('Union3D(SymRef3D,Move3D)', 3), final loss: 0.5689
Total nodes replaced: 41, avg per shape: 0.27
Shape 1 contains 1 abstraction nodes
Shape 4 contains 1 abstraction nodes
Shape 5 contains 1 abstraction nodes
Shape 7 contains 1 abstraction nodes
Shape 9 contains 1 abstraction nodes
Shape 11 contains 1 abstraction nodes
Shape 15 contains 1 abstraction nodes
Shape 20 contains 1 abstraction nodes
Shape 21 contains 1 abstraction nodes
Shape 26 contains 1 abstraction nodes
Shape 28 contains 1 abstraction nodes
Shape 32 contains 1 abstraction nodes
Shape 33 contains 1 abstraction nodes
Shape 42 contains 1 abstraction nodes
Sh

In [72]:
original = dataset[0]
abstracted = abstracted_dataset[0]

print("Original shape")
visualize_boxes_3d(original.get_box3d_list())

print("Abstracted shape")
visualize_boxes_3d(abstracted.get_box3d_list())

for node in abstracted_dataset:
    if isinstance(node, AbstractionNode):
        print(node)

Original shape


Output()

Abstracted shape


Output()

AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.9279893636703491, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.971859335899353, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.9638319611549377, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.9447119235992432, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.8755001425743103, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 1.0119118690490723, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.9011883735656738, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.9536804556846619, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 1.001548409461975, 0.0])
AbstractionNode(type_list=['Union3D', 'Move3D', 'Union3D'], latent=[0.0, 0.9234377145767212, 0.0])
AbstractionN

In [79]:
abstracted_dataset[3]